# Task 5.2 — Agent Integration and End-to-End Assessment

## Objective
Integrate the agent with the project's **real tool layer**, execute an assessment for one firm-year, and save the assessment plus its execution trace for Task 5.3.

This notebook does not generate substitute assessment data. It checks the supplied tool implementations before running. If the tools still return stub values, it stops with a clear message rather than presenting invented results as real evidence.

**Project rules retained:** AUOPIC=2 is the adverse ICFR opinion; AUOPIC=1 is effective; AUOPIC=0 is not assessed/unknown. The forecast target is the following fiscal year's opinion. A predicted adverse opinion is not proof of fraud or a financial misstatement.

## 1. Imports and user inputs
Set the firm identifier and feature year only after confirming that the corresponding firm-year exists in the approved panel.

In [ ]:
from pathlib import Path
import os, json, copy, inspect, importlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

FIRM_ID = "001004"       # Replace with a valid six-character gvkey from the approved panel
FEATURE_YEAR = 2018      # Replace with the fiscal year being assessed
MAX_TOOL_CALLS = 8
OUTPUT_DIR = Path("task_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

assert isinstance(FIRM_ID, str) and len(FIRM_ID) == 6, "gvkey must be a six-character string with leading zeros preserved."
assert isinstance(FEATURE_YEAR, int)
print(f"Assessment request: gvkey={FIRM_ID}, feature year={FEATURE_YEAR}, outcome year={FEATURE_YEAR + 1}")

## 2. Implementation readiness audit
This checks the files and model/data configuration expected by the project. The decision log specifies `config.PANEL_FILE` as the only permitted panel input and places the canonical panel and versioned model in the shared `data/` and `models/` locations.

In [ ]:
required_files = ["agent.py", "tools.py", "model.py", "scoring.py", "data.py", "config.py"]
file_audit = pd.DataFrame([
    {"file": name, "present": Path(name).exists()}
    for name in required_files
])
display(file_audit)

missing_files = file_audit.loc[~file_audit["present"], "file"].tolist()
if missing_files:
    print("Readiness issue: required project modules are missing:", missing_files)

try:
    import config
    print("Configured panel:", config.PANEL_FILE)
    print("Configured model directory:", config.MODEL_DIR)
    print("Configured seed:", getattr(config, "SEED", "not declared"))
    if not Path(config.PANEL_FILE).exists():
        print("Readiness issue: config.PANEL_FILE does not exist in this runtime.")
except Exception as exc:
    config = None
    print("Readiness issue: config.py could not be imported:", repr(exc))

try:
    import tools
    import agent
    print("Agent and tool modules imported.")
except Exception as exc:
    raise RuntimeError(
        "Cannot import the project agent/tools. Run this notebook from the project root "
        "with the agreed project modules available."
    ) from exc

## 3. Check whether the nine tools are real implementations
The existing `tools.py` supplied with this workspace labels its returns as stubs. The check below calls each tool with a safe probe and inspects its source/notes. It does not treat a successful Python call as proof that the returned evidence is real.

In [ ]:
def tool_probe(name):
    fn = getattr(tools, name, None)
    if fn is None:
        return {"tool": name, "available": False, "stub_signal": True, "source": None, "notes": "Function missing"}
    sig = inspect.signature(fn)
    # Probe arguments follow the current project tool function signatures.
    args = {
        "get_firm_history": (FIRM_ID, FEATURE_YEAR),
        "get_control_history": (FIRM_ID,),
        "score_firm": (FIRM_ID, FEATURE_YEAR),
        "get_percentile": (FIRM_ID, FEATURE_YEAR, "ROA"),
        "get_deviation": (FIRM_ID, FEATURE_YEAR, "ROA"),
        "get_peer_cohort": (FIRM_ID, FEATURE_YEAR),
        "get_cohort_rate": ("all", FEATURE_YEAR),
        "get_industry_context": (FIRM_ID, FEATURE_YEAR),
        "get_filing_excerpt": (FIRM_ID, FEATURE_YEAR, "item9a"),
    }[name]
    try:
        result = fn(*args)
        source = result.get("source") if isinstance(result, dict) else None
        notes = result.get("notes") if isinstance(result, dict) else str(type(result))
        signal = "stub" in str(source).lower() or "stub" in str(notes).lower()
        return {"tool": name, "available": True, "stub_signal": signal, "source": source, "notes": notes}
    except Exception as exc:
        return {"tool": name, "available": True, "stub_signal": None, "source": None, "notes": f"Probe error: {type(exc).__name__}: {exc}"}

tool_names = [
    "get_firm_history", "get_control_history", "score_firm", "get_percentile",
    "get_deviation", "get_peer_cohort", "get_cohort_rate",
    "get_industry_context", "get_filing_excerpt"
]
tool_audit = pd.DataFrame([tool_probe(name) for name in tool_names])
display(tool_audit[["tool", "available", "stub_signal", "source", "notes"]])

stub_tools = tool_audit.loc[tool_audit["stub_signal"] == True, "tool"].tolist()
missing_tools = tool_audit.loc[tool_audit["available"] == False, "tool"].tolist()
print("Tools showing stub signals:", stub_tools)
print("Missing tools:", missing_tools)

## 4. Stop safely if the real integration prerequisites are not met
The real run requires: (a) real implementations for the ungated tools, (b) a configured canonical panel and model artifact, (c) a working LLM provider, and (d) resolved temporal/no-look-ahead issues. The filing tool is conditional on its separate coverage gate.

In [ ]:
# The filing excerpt tool is gated in the agreed contract; it is not required for a panel/model-only run.
required_for_run = [name for name in tool_names if name != "get_filing_excerpt"]
unready = [
    row["tool"] for row in tool_audit.to_dict("records")
    if row["tool"] in required_for_run and (not row["available"] or row["stub_signal"] is True)
]
if unready:
    raise RuntimeError(
        "REAL END-TO-END RUN BLOCKED. These required tools are missing or still stubs: "
        + ", ".join(unready)
        + ". Replace them with panel/model-backed implementations conforming to TOOL_CONTRACT.md, "
          "then rerun this notebook. No illustrative assessment has been generated."
    )

if config is None:
    raise RuntimeError("REAL END-TO-END RUN BLOCKED: config.py is not available.")

panel_path = Path(config.PANEL_FILE)
if not panel_path.exists():
    raise FileNotFoundError(
        f"Canonical panel configured in config.PANEL_FILE is unavailable: {panel_path}. "
        "Do not substitute `final_panel 1.csv`; the decision log explicitly disallows that duplicate."
    )

model_path = Path(config.MODEL_DIR) / "financial_only_xgb_v2.pkl"
if not model_path.exists():
    raise FileNotFoundError(f"Expected frozen model artifact is unavailable: {model_path}")

print("Preflight passed. Proceed only after the model/tool temporal checks below are cleared.")

## 5. Temporal and contract checks to complete before the live run
The supplied `model.py` and `scoring.py` should not be assumed safe merely because the artifact loads. The current visible code computes `cohort_base_rate` from the outcome year (t+1), and the scoring prior-opinion branch assigns current-year AUOPIC rather than a lagged prior-year opinion. These are material issues to resolve against the latest team decisions before an assessment can be called as-of-year-t.

In [ ]:
temporal_review = pd.DataFrame([
    {"component": "score_firm cohort_base_rate", "observed_in_supplied_code": "Computed from assessed firms in fyear + 1", "required_action": "Remove future-year outcome information from the t-year tool response; confirm allowed historical baseline with project owner."},
    {"component": "prior-opinion branch in scoring.py", "observed_in_supplied_code": "Sets prior opinion from current row auopic", "required_action": "Use a correctly lagged prior-year opinion only if the selected model contract requires it; otherwise keep financial-only artifact."},
    {"component": "peer cohort rate", "observed_in_supplied_code": "Must be checked against latest amended decision", "required_action": "Use the agreed same-year peer rate and avoid future outcome leakage."},
    {"component": "firm/control history", "observed_in_supplied_code": "Agent schema requires through_fyear", "required_action": "Ensure every history query is capped at FEATURE_YEAR."},
])
display(temporal_review)
display(Markdown(
    "**Interpretation:** these are not cosmetic issues. If future-year outcomes are returned as evidence for a forecast made in year t, the agent can appear more accurate than a genuine as-of-year-t system. Resolve and document these points before enabling the run cell."
))

## 6. Configure the LLM adapter and tool signatures
The agent's tool schemas and the current tool contract have some parameter-name differences. The wrappers below translate agent-facing names to the project tool functions. The OpenAI-compatible adapter requires an API key in the environment; no key is embedded in the notebook.

In [ ]:
# Install if needed: %pip install openai
from openai import OpenAI

class OpenAICompatibleLLM:
    def __init__(self, model_name=None):
        self.client = OpenAI()
        self.model_name = model_name or os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

    def complete(self, messages, tool_schemas):
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=messages,
            tools=tool_schemas,
            tool_choice="auto",
            temperature=0,
        )
        message = response.choices[0].message
        if message.tool_calls:
            call = message.tool_calls[0]
            return {
                "type": "tool_call",
                "name": call.function.name,
                "arguments": json.loads(call.function.arguments or "{}"),
            }
        return {"type": "final", "content": message.content or "{}"}

# Adapt agent schema names to the contract/tool signatures.
def agent_tool_map():
    return {
        "get_firm_history": lambda gvkey, through_fyear: tools.get_firm_history(gvkey, through_fyear),
        "get_control_history": lambda gvkey, through_fyear: tools.get_control_history(gvkey),
        "score_firm": lambda gvkey, feature_year: tools.score_firm(gvkey, feature_year),
        "get_percentile": lambda gvkey, feature_year, variable: tools.get_percentile(gvkey, feature_year, variable),
        "get_deviation": lambda gvkey, feature_year, variable: tools.get_deviation(gvkey, feature_year, variable),
        "get_peer_cohort": lambda gvkey, feature_year, k=200: tools.get_peer_cohort(gvkey, feature_year, k),
        "get_cohort_rate": lambda cohort_key, cohort_value, feature_year: tools.get_cohort_rate(f"{cohort_key}_{cohort_value}", feature_year),
        "get_industry_context": lambda sic, feature_year: tools.get_industry_context(FIRM_ID, feature_year),
        "get_filing_excerpt": lambda gvkey, fyear, section: tools.get_filing_excerpt(gvkey, fyear, section),
    }

print("LLM adapter and contract wrappers defined. Set OPENAI_API_KEY before running the live assessment.")

## 7. Execute the real assessment
This cell is intentionally gated. It runs only after the readiness audit has passed and an LLM key is configured.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in your environment before running the live assessment.")

from agent import run_agent
llm = OpenAICompatibleLLM()
run = run_agent(
    firm_id=FIRM_ID,
    feature_year=FEATURE_YEAR,
    user_question=(
        "Assess the risk of an adverse ICFR opinion in the following fiscal year. "
        "Use the available tools, distinguish historical facts from the forecast, "
        "report contradictory evidence, and cite every factual claim."
    ),
    llm=llm,
    tools=agent_tool_map(),
    max_tool_calls=MAX_TOOL_CALLS,
)

raw_assessment = copy.deepcopy(run.output)
execution_trace = copy.deepcopy(run.trace)
print("Run status:", run.status)
print("Run ID:", run.run_id)
print("Tool calls:", run.tool_calls_used)
display(pd.json_normalize(raw_assessment.get("assessment", {})))
display(Markdown("### Assessment summary\n" + raw_assessment.get("assessment", {}).get("summary", "No summary returned.")))

## 8. Inspect the actual tool execution trace

In [ ]:
trace_rows = []
for event in execution_trace:
    if event.get("event") == "tool_call":
        result = event.get("result") or {}
        trace_rows.append({
            "tool": event.get("tool"),
            "status": event.get("status"),
            "result_id": result.get("result_id"),
            "source": result.get("source"),
            "notes": result.get("notes"),
        })
trace_df = pd.DataFrame(trace_rows)
display(trace_df)

if not trace_df.empty:
    ax = trace_df["tool"].value_counts().plot(kind="bar", figsize=(9,4), color="#3478A8")
    ax.set_title("Actual tool calls in the assessment run")
    ax.set_xlabel("Tool")
    ax.set_ylabel("Calls")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()
    display(Markdown(
        "**Interpretation:** each bar counts how many times the agent called that tool. "
        "The trace table is the audit trail: it records whether each call succeeded and "
        "the source/notes returned by the tool. A tool-call count alone does not establish evidence quality."
    ))

## 9. Save the Task 5.2 handoff for Task 5.3

In [ ]:
handoff = {
    "task": "5.2",
    "run_id": run.run_id,
    "run_status": run.status,
    "firm_id": FIRM_ID,
    "feature_year": FEATURE_YEAR,
    "outcome_year": FEATURE_YEAR + 1,
    "tool_calls_used": run.tool_calls_used,
    "assessment_raw": raw_assessment,
    "trace": execution_trace,
    "execution_mode": "REAL_TOOLS_AND_LLM",
    "panel_path": str(config.PANEL_FILE),
    "model_path": str(model_path),
}
handoff_path = OUTPUT_DIR / "task_5_2_run.json"
handoff_path.write_text(json.dumps(handoff, indent=2, default=str), encoding="utf-8")
print("Saved Task 5.3 handoff:", handoff_path.resolve())

## 10. Interpretation and completion checklist

A successful Task 5.2 run should leave three inspectable outputs: a structured assessment, a trace of the tools actually called, and a saved JSON handoff. The assessment should distinguish the forecast for year t+1 from facts observed by year t.

Before treating the run as complete, confirm:
- all non-gated tools return panel/model-backed values and contract-compliant envelopes;
- the panel and model paths are the approved canonical paths;
- the model/tool outputs contain no future-year information unavailable at t;
- missing values are returned as `None` with explanatory notes, never invented defaults;
- the trace preserves raw tool results and unique result IDs.

If the preflight stops, that is a genuine integration blocker—not a failed assessment and not a reason to substitute invented values.